# CallWhisper-8k: Clean Colab Benchmark v2

This is the canonical **Run all** notebook for the fixed GramVaani 100-file benchmark. It runs three models on the same 100 recordings:

- OpenAI Whisper medium
- OpenAI Whisper large-v3
- ARTPARK-IISc Whisper medium Vaani Hindi

The notebook derives the native 8 kHz and higher-source-rate metrics from those same predictions, so each recording is transcribed only once per model. Results are checkpointed to Google Drive after every model.

Before running: select **Runtime > Change runtime type > T4 GPU**, then choose **Runtime > Run all**.

In [ ]:
# Environment, GPU, and Drive setup. This cell always restores a valid working directory.
import csv
import importlib.metadata
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path
from statistics import mean

import torch
from google.colab import drive

CONTENT_DIR = Path('/content')
REPO_DIR = CONTENT_DIR / 'CallWhisper-8k'
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/call-whisper')
DRIVE_DATA_DIR = DRIVE_PROJECT_DIR / 'GV_Dev_5h'
DRIVE_RESULTS_DIR = DRIVE_PROJECT_DIR / 'results' / 'benchmark_v2'
REPO_URL = 'https://github.com/anshulLuhsna/CallWhisper-8k.git'
SEED = 0
RESUME_FROM_DRIVE = True

CONTENT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(CONTENT_DIR)
drive.mount('/content/drive')
subprocess.run(['nvidia-smi'], check=True)
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. In Colab select Runtime > Change runtime type > T4 GPU.')
print('Working directory:', Path.cwd())
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Clone from a stable parent directory, then install only the evaluation dependencies.
os.chdir(CONTENT_DIR)
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)],
    cwd=CONTENT_DIR,
    check=True,
)
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        '-e', '.',
        'transformers>=4.46,<5',
        'accelerate>=1,<2',
        'librosa>=0.10,<1',
    ],
    cwd=REPO_DIR,
    check=True,
)
os.chdir(REPO_DIR)
os.environ['PYTHONPATH'] = str(REPO_DIR / 'src')
COMMIT = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True
).strip()
print('Repository:', REPO_DIR)
print('Commit:', COMMIT)


In [ ]:
# Link the Drive audio and validate the complete benchmark before loading any model.
if not DRIVE_DATA_DIR.exists():
    raise FileNotFoundError(
        f'Missing {DRIVE_DATA_DIR}. Expected MyDrive/call-whisper/GV_Dev_5h.'
    )
for required in [DRIVE_DATA_DIR / 'Audio', DRIVE_DATA_DIR / 'text', DRIVE_DATA_DIR / 'mp3.scp']:
    if not required.exists():
        raise FileNotFoundError(f'Incomplete GV_Dev_5h dataset; missing {required}')

repo_data = REPO_DIR / 'datasets' / 'GV_Dev_5h'
if repo_data.is_symlink() or repo_data.is_file():
    repo_data.unlink()
elif repo_data.exists():
    shutil.rmtree(repo_data)
repo_data.symlink_to(DRIVE_DATA_DIR, target_is_directory=True)

MANIFESTS = {
    'gramvaani_dev_100': REPO_DIR / 'datasets/manifests/gramvaani_dev_100.csv',
    'gramvaani_dev_100_8khz': REPO_DIR / 'datasets/manifests/gramvaani_dev_100_8khz.csv',
    'gramvaani_dev_100_highrate': REPO_DIR / 'datasets/manifests/gramvaani_dev_100_highrate.csv',
}

def read_manifest(path):
    with path.open(encoding='utf-8', newline='') as handle:
        return list(csv.DictReader(handle))

manifest_rows = {name: read_manifest(path) for name, path in MANIFESTS.items()}
manifest_ids = {
    name: {Path(row['audio_path']).name for row in rows}
    for name, rows in manifest_rows.items()
}
for name, rows in manifest_rows.items():
    missing = [row['audio_path'] for row in rows if not (REPO_DIR / row['audio_path']).exists()]
    print(f'{name}: rows={len(rows)}, missing_audio={len(missing)}')
    if missing:
        raise FileNotFoundError(f'{name} is missing audio; first missing file: {missing[0]}')

all_ids = manifest_ids['gramvaani_dev_100']
khz8_ids = manifest_ids['gramvaani_dev_100_8khz']
highrate_ids = manifest_ids['gramvaani_dev_100_highrate']
assert len(all_ids) == 100, f'Expected 100 files, found {len(all_ids)}'
assert len(khz8_ids) == 56, f'Expected 56 native 8 kHz files, found {len(khz8_ids)}'
assert len(highrate_ids) == 44, f'Expected 44 high-rate files, found {len(highrate_ids)}'
assert not (khz8_ids & highrate_ids), '8 kHz and high-rate manifests overlap'
assert khz8_ids | highrate_ids == all_ids, 'Rate splits do not reconstruct the 100-file slice'
print('Dataset and all three manifests validated.')


In [ ]:
# Model plan. Each model transcribes the 100-file manifest exactly once.
LOCAL_RESULTS_DIR = REPO_DIR / 'results' / 'benchmark_v2'
LOCAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RUNS = [
    {
        'kind': 'openai',
        'model': 'medium',
        'prefix': 'colab_whisper_medium',
    },
    {
        'kind': 'openai',
        'model': 'large-v3',
        'prefix': 'colab_whisper_large_v3',
    },
    {
        'kind': 'hf',
        'model': 'ARTPARK-IISc/whisper-medium-vaani-hindi',
        'prefix': 'colab_hf_artpark_iisc_whisper_medium_vaani_hindi',
    },
]

def result_path(run, slice_name, root=LOCAL_RESULTS_DIR):
    return root / f"{run['prefix']}_{slice_name}_seed{SEED}.json"

print('Planned model runs:', len(RUNS))
for run in RUNS:
    print('-', run['model'])


In [ ]:
# Run inference with live tqdm output. Completed 100-file results can resume from Drive.
def valid_complete_result(path):
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
        return payload['summary']['num_files'] == 100 and len(payload['samples']) == 100
    except (FileNotFoundError, KeyError, TypeError, ValueError, json.JSONDecodeError):
        return False

env = {**os.environ, 'PYTHONPATH': str(REPO_DIR / 'src'), 'PYTHONUNBUFFERED': '1'}
full_manifest = str(MANIFESTS['gramvaani_dev_100'].relative_to(REPO_DIR))

for index, run in enumerate(RUNS, start=1):
    local_output = result_path(run, 'gramvaani_dev_100')
    drive_output = result_path(run, 'gramvaani_dev_100', DRIVE_RESULTS_DIR)
    print('\n' + '=' * 88, flush=True)
    print(f"MODEL {index}/{len(RUNS)}: {run['model']}", flush=True)

    if RESUME_FROM_DRIVE and valid_complete_result(drive_output):
        shutil.copy2(drive_output, local_output)
        print('Loaded completed result from Drive:', drive_output, flush=True)
        continue

    if run['kind'] == 'openai':
        cmd = [
            sys.executable, '-m', 'callwhisper.eval',
            '--manifest', full_manifest,
            '--model', run['model'],
            '--language-mode', 'manifest',
            '--seed', str(SEED),
            '--output-json', str(local_output),
        ]
    else:
        cmd = [
            sys.executable, '-m', 'callwhisper.eval.hf_runner',
            '--manifest', full_manifest,
            '--model-id', run['model'],
            '--language-mode', 'manifest',
            '--seed', str(SEED),
            '--output-json', str(local_output),
        ]

    print('Command:', ' '.join(cmd), flush=True)
    started = time.time()
    subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)
    if not valid_complete_result(local_output):
        raise RuntimeError(f'Run finished without a valid 100-file result: {local_output}')
    shutil.copy2(local_output, drive_output)
    print(f'Saved checkpoint to Drive after {(time.time() - started) / 60:.1f} minutes:', drive_output)


In [ ]:
# Derive 8 kHz and high-rate result files from the same 100 predictions.
def summarize(samples):
    if not samples:
        raise ValueError('Cannot summarize an empty slice')
    return {
        'num_files': len(samples),
        'wer': mean(sample['wer'] for sample in samples),
        'cer': mean(sample['cer'] for sample in samples),
    }

for run in RUNS:
    full_path = result_path(run, 'gramvaani_dev_100')
    full_payload = json.loads(full_path.read_text(encoding='utf-8'))
    sample_by_name = {Path(sample['audio_path']).name: sample for sample in full_payload['samples']}
    if set(sample_by_name) != all_ids:
        raise RuntimeError(f"Prediction IDs do not match the frozen manifest for {run['model']}")

    for slice_name, ids in manifest_ids.items():
        condition = manifest_rows[slice_name][0]['condition']
        samples = [
            {**sample_by_name[name], 'slice': slice_name, 'condition': condition}
            for name in sorted(ids)
        ]
        payload = {'summary': summarize(samples), 'samples': samples}
        local_path = result_path(run, slice_name)
        drive_path = result_path(run, slice_name, DRIVE_RESULTS_DIR)
        local_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
        shutil.copy2(local_path, drive_path)
        print(run['model'], slice_name, payload['summary'])


In [ ]:
# Build report-ready Markdown, JSON, and CSV and save reproducibility metadata.
import pandas as pd

comparison_rows = []
for run in RUNS:
    for slice_name in MANIFESTS:
        path = result_path(run, slice_name)
        payload = json.loads(path.read_text(encoding='utf-8'))
        summary = payload['summary']
        sample = payload['samples'][0]
        comparison_rows.append({
            'model': sample['model'],
            'slice': slice_name,
            'condition': sample['condition'],
            'files': summary['num_files'],
            'wer': round(summary['wer'], 4),
            'cer': round(summary['cer'], 4),
            'file': path.name,
        })

comparison = pd.DataFrame(comparison_rows).sort_values(['slice', 'wer']).reset_index(drop=True)
md_path = REPO_DIR / 'results' / 'model_comparison_v2.md'
json_path = REPO_DIR / 'results' / 'model_comparison_v2.json'
csv_path = REPO_DIR / 'results' / 'model_comparison_v2.csv'
md_path.write_text('# Model Comparison v2\n\n' + comparison.to_markdown(index=False) + '\n', encoding='utf-8')
json_path.write_text(comparison.to_json(orient='records', force_ascii=False, indent=2) + '\n', encoding='utf-8')
comparison.to_csv(csv_path, index=False)

metadata = {
    'repo_commit': COMMIT,
    'seed': SEED,
    'gpu': torch.cuda.get_device_name(0),
    'python': sys.version,
    'torch': torch.__version__,
    'transformers': importlib.metadata.version('transformers'),
    'openai_whisper': importlib.metadata.version('openai-whisper'),
    'models': [run['model'] for run in RUNS],
    'manifests': {name: len(rows) for name, rows in manifest_rows.items()},
}
metadata_path = REPO_DIR / 'results' / 'model_comparison_v2_run_metadata.json'
metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

for path in [md_path, json_path, csv_path, metadata_path]:
    shutil.copy2(path, DRIVE_RESULTS_DIR / path.name)

display(comparison)
print('\nComplete. All outputs are saved in:', DRIVE_RESULTS_DIR)


## Interpretation

Report the fixed-slice result directly: *On the fixed GramVaani 100-file slice, model X obtained WER A and CER B. On the native 8 kHz subset, it obtained WER C and CER D.*

These results compare models under one reproducible benchmark pipeline. They are not evidence of a global Hindi ASR leaderboard.